# 1. XOR: gradient descent vs. NEAT

XOR outputs 1 only when exactly one input is 1. It is the smallest problem that needs a hidden neuron. Gradient descent gets a shape we choose; NEAT starts with no hidden neurons and has to grow one.

Part of Kyle Robin's DCS 340 Project II on NEAT (Stanley & Miikkulainen, 2002). Run the cells top to bottom with **Shift + Enter**.

In [ ]:
# Run this first: installs NEAT (takes a few seconds)
!pip install -q neat-python

In [ ]:
# Setup: imports and shared helper code. Run this cell first.
import os, time, math, tempfile, contextlib, io, warnings
import numpy as np
import matplotlib.pyplot as plt
import neat

warnings.filterwarnings("ignore")
SEED = 0  # change this to see how much results depend on luck


# ---------- Gradient descent: a small network written from scratch ----------
def sigmoid(z):
    return 1 / (1 + np.exp(-z))


class GDNet:
    """A network with ONE hidden layer whose size YOU choose.
    It learns with plain gradient descent + backpropagation."""

    def __init__(self, n_inputs=2, n_hidden=4, seed=0):
        rng = np.random.default_rng(seed)
        self.W1 = rng.normal(0, 1, (n_inputs, n_hidden))  # input -> hidden weights
        self.b1 = np.zeros(n_hidden)                       # hidden biases
        self.W2 = rng.normal(0, 1, (n_hidden, 1))          # hidden -> output weights
        self.b2 = np.zeros(1)                              # output bias

    def forward(self, X):
        """Forward pass: weighted sum + sigmoid, layer by layer."""
        self.h = sigmoid(X @ self.W1 + self.b1)
        self.out = sigmoid(self.h @ self.W2 + self.b2)
        return self.out[:, 0]

    def train(self, X, y, lr=2.0, epochs=5000):
        """Gradient descent on mean squared error. Returns the loss at every step."""
        losses = []
        for _ in range(epochs):
            p = self.forward(X)
            err = p - y
            losses.append(np.mean(err ** 2))
            # Backpropagation: pass the error backward with the chain rule
            d_out = (2 * err / len(y) * p * (1 - p))[:, None]
            gW2, gb2 = self.h.T @ d_out, d_out.sum(0)
            d_h = d_out @ self.W2.T * self.h * (1 - self.h)
            gW1, gb1 = X.T @ d_h, d_h.sum(0)
            # Step downhill: w_new = w_old - learning_rate * slope
            self.W2 -= lr * gW2; self.b2 -= lr * gb2
            self.W1 -= lr * gW1; self.b1 -= lr * gb1
        return losses

    def n_params(self):
        return self.W1.size + self.b1.size + self.W2.size + self.b2.size

    def predict(self, X):
        return self.forward(np.asarray(X, float))


# ---------- NEAT: settings and a runner ----------
NEAT_SETTINGS = {
    "num_inputs": 2, "pop_size": 150, "fitness_threshold": 3.9,
    "node_add_prob": 0.2, "conn_add_prob": 0.5, "weight_mutate_power": 0.5,
    "activation_options": "sigmoid", "activation_mutate_rate": 0.0,
    "compatibility_threshold": 3.0,
}

NEAT_TEMPLATE = """
[NEAT]
fitness_criterion = max
fitness_threshold = {fitness_threshold}
pop_size = {pop_size}
reset_on_extinction = True
no_fitness_termination = False
[DefaultGenome]
num_inputs = {num_inputs}
num_hidden = 0
num_outputs = 1
initial_connection = full
feed_forward = True
activation_default = sigmoid
activation_mutate_rate = {activation_mutate_rate}
activation_options = {activation_options}
aggregation_default = sum
aggregation_mutate_rate = 0.0
aggregation_options = sum
bias_init_mean = 0.0
bias_init_stdev = 1.0
bias_max_value = 30.0
bias_min_value = -30.0
bias_mutate_power = {weight_mutate_power}
bias_mutate_rate = 0.7
bias_replace_rate = 0.1
response_init_mean = 1.0
response_init_stdev = 0.0
response_max_value = 30.0
response_min_value = -30.0
response_mutate_power = 0.0
response_mutate_rate = 0.0
response_replace_rate = 0.0
weight_init_mean = 0.0
weight_init_stdev = 1.0
weight_max_value = 30
weight_min_value = -30
weight_mutate_power = {weight_mutate_power}
weight_mutate_rate = 0.8
weight_replace_rate = 0.1
enabled_default = True
enabled_mutate_rate = 0.01
conn_add_prob = {conn_add_prob}
conn_delete_prob = 0.5
node_add_prob = {node_add_prob}
node_delete_prob = 0.2
compatibility_disjoint_coefficient = 1.0
compatibility_weight_coefficient = 0.5
[DefaultSpeciesSet]
compatibility_threshold = {compatibility_threshold}
[DefaultStagnation]
species_fitness_func = max
max_stagnation = 20
species_elitism = 2
[DefaultReproduction]
elitism = 2
survival_threshold = 0.2
"""


def run_neat(fitness_fn, changes=None, generations=300, seed=0):
    """Evolve networks with NEAT. fitness_fn(net) -> a score (higher is better).
    Returns (best genome, its network, config, stats, seconds)."""
    settings = dict(NEAT_SETTINGS, **(changes or {}))
    path = os.path.join(tempfile.gettempdir(), "neat_compare.ini")
    with open(path, "w") as f:
        f.write(NEAT_TEMPLATE.format(**settings))
    config = neat.Config(neat.DefaultGenome, neat.DefaultReproduction,
                         neat.DefaultSpeciesSet, neat.DefaultStagnation, path)

    def eval_genomes(genomes, config):
        for _, g in genomes:
            g.fitness = fitness_fn(neat.nn.FeedForwardNetwork.create(g, config))

    pop = neat.Population(config, seed=seed)
    stats = neat.StatisticsReporter()
    pop.add_reporter(stats)
    start = time.time()
    with contextlib.redirect_stdout(io.StringIO()):
        winner = pop.run(eval_genomes, generations)
    net = neat.nn.FeedForwardNetwork.create(winner, config)
    return winner, net, config, stats, time.time() - start


def neat_size(winner, config):
    """(hidden neurons, enabled connections, total parameters = weights + biases)"""
    hidden = sum(1 for k in winner.nodes if k not in config.genome_config.output_keys)
    conns = sum(c.enabled for c in winner.connections.values())
    return hidden, conns, conns + len(winner.nodes)


def neat_predict(net, X):
    return np.array([net.activate(x)[0] for x in X])


def accuracy(pred, y):
    return np.mean((pred > 0.5) == y)

print("Ready.")


## Gradient descent

In [ ]:
XOR_X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], float)
XOR_Y = np.array([0, 1, 1, 0], float)

# Gradient descent: WE choose the shape (2 hidden neurons), it tunes the weights
start = time.time()
gd_xor = GDNet(n_hidden=2, seed=SEED)
gd_losses = gd_xor.train(XOR_X, XOR_Y, lr=2.0, epochs=5000)
gd_time = time.time() - start

print("Gradient descent outputs:", np.round(gd_xor.predict(XOR_X), 3))
print(f"Accuracy {accuracy(gd_xor.predict(XOR_X), XOR_Y):.0%}, "
      f"{gd_xor.n_params()} parameters, {gd_time:.2f} s")

## NEAT

In [ ]:
# NEAT: starts with NO hidden neurons and has to find the shape itself.
# Fitness = 4 minus squared error on the 4 cases (4 = perfect), same as xor_neat.py
def xor_fitness(net):
    return 4 - np.sum((neat_predict(net, XOR_X) - XOR_Y) ** 2)

w, neat_xor, cfg, stats, neat_time = run_neat(xor_fitness, generations=300, seed=SEED)
hidden, conns, params = neat_size(w, cfg)

print("NEAT outputs:", np.round(neat_predict(neat_xor, XOR_X), 3))
print(f"Accuracy {accuracy(neat_predict(neat_xor, XOR_X), XOR_Y):.0%}, "
      f"{hidden} hidden neuron(s), {params} parameters, "
      f"{len(stats.most_fit_genomes)} generations, {neat_time:.2f} s")

## How each method improved

In [ ]:
# How each method improved over time
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(gd_losses)
axes[0].set(title="Gradient descent: loss goes DOWN", xlabel="training step", ylabel="mean squared error")
axes[1].plot([g.fitness for g in stats.most_fit_genomes])
axes[1].set(title="NEAT: best fitness goes UP", xlabel="generation", ylabel="fitness (4 = perfect)")
plt.tight_layout(); plt.show()

**Your notes:**

- 
